# EcoConsensus: fine-tuning method comparison (IndicBERT and MuRIL)

Compares **head only**, **head + last N layers**, **LoRA** and **QLoRA** (4-bit, optionally 8-bit) on climate-stance classification with sentence-pair input `(claim, comment)`. Labels: **0 = favour, 1 = neutral, 2 = against**. Full fine-tuning is intentionally not included.

**Two stages.** Stage 1 searches the LoRA hyperparameters (rank `r`, `alpha`, target modules, learning rate) over the grid in `LORA_SEARCH`, scoring each combination on `val`. Stage 2 runs every method with all `SEEDS`; LoRA and QLoRA use the combination stage 1 picked for that model (QLoRA reuses LoRA's by default, so the only difference between them is quantization).

**Data:** a small subset of **SemEval-2016 Task 6** (climate tweets) in English + 3 Hindi variants (pure, Hinglish, code-mixed) + 3 Bengali variants (pure, Banglish, code-mixed), built on the Mac by `experiment/prepare_data.py` as `experiment_data.zip`. The notebook unzips it and splits it **by tweet** into `train` / `val` / `test`, so all 7 versions of a tweet stay in the same split.

**How to run (Colab):** upload `experiment_data.zip` to `MyDrive/EcoConsensus/`, then Runtime > Change runtime type > **T4 GPU**, then **Runtime > Run all**. Everything is saved to Google Drive (`MyDrive/EcoConsensus/experiment_outputs/`): per-run results, predictions, frozen splits, the MLflow database (`mlflow.db` + `mlartifacts/`) and timestamped `.md` reports. If the session disconnects, reconnect and Run all again: finished runs are skipped.

Read `SETUP.md` first. Only cell 1 needs editing.

| Cell | What it does |
|---|---|
| 1 | Config (paths, models, methods, LoRA search grid, split fractions, seeds, hyperparameters) |
| 2 | Installs libraries, mounts Drive, picks device (Apple MPS, then CUDA GPU, then CPU) |
| 3 | Helpers: preprocessing, metrics |
| 4 | Unzips the data bundle, builds frozen, leak-free train / val / test splits |
| 5 | Downloads tokenizers, prints token-length and `[UNK]` stats |
| 6 | Training code for the methods (QLoRA: 4-bit NF4 or 8-bit base) |
| 7 | TF-IDF and Majority reference baselines |
| 8 | Stage 1 LoRA search, then stage 2 method comparison (resumable), logs to MLflow on Drive |
| 9 | Writes a new timestamped report (plain Python, no AI) |

## 1. Config

In [1]:
# ============================ 1. CONFIG - the only cell you normally edit ============================
EXPERIMENT_NAME = "finetune_method_comparison"

# ---- Where things live ----
DRIVE_FOLDER   = "EcoConsensus"        # Colab: MyDrive/<DRIVE_FOLDER>/ holds experiment_data.zip and receives all outputs
DATA_ZIP_NAME  = "experiment_data.zip" # made by experiment/prepare_data.py
LOCAL_DATA_DIR = None                  # Mac/local only: folder containing semeval.csv + manifest.json
LOCAL_OUT_DIR  = None                  # Mac/local only: output folder (default ./experiment_outputs)

# ---- Environment ----
INSTALL_DEPS       = True    # pip install -U transformers peft accelerate mlflow ... (set False if already installed)
ALLOW_CPU_ON_CLOUD = False   # on Colab/Kaggle, stop if no GPU is found instead of silently training on CPU
USE_MLFLOW         = True    # logs every run to <outputs>/mlflow.db (open later with `mlflow ui`)

# ---- Models (Hugging Face ids) ----
MODELS = {
    "muril":     "google/muril-base-cased",
    "indicbert": "ai4bharat/IndicBERTv2-MLM-only",   # IndicBERT v2 family; change here if you pick another checkpoint
}

# ---- Fine-tuning methods to compare (full fine-tuning is intentionally excluded) ----
RUN_METHODS = ["head_only", "last_n", "lora", "qlora"]   # delete names to skip a method
LAST_N_LIST = [2, 4]                                      # "head + last N layers": one run per N
# Fixed learning rates for the non-LoRA methods. head_lr (optional) applies to the classifier/pooler.
LR = {
    "head_only": {"lr": 1e-3},
    "last_n":    {"lr": 5e-5, "head_lr": 5e-4},
}

# ---- LoRA / QLoRA hyperparameters: searched, not fixed ----
# Stage 1 (search): every combination below is trained with SEARCH_SEEDS and scored on val Macro-F1. Per model, the
#   best combination is taken, then (parsimony) the one with the fewest trainable parameters within max(1 std, 0.010) of it.
# Stage 2 (compare): the picked combination is re-run with all SEEDS and compared with the other methods.
LORA_SEARCH = {
    "r":          [8, 16, 32],
    "alpha_mult": [1, 2],                 # lora_alpha = r * alpha_mult, i.e. the update scale alpha/r is 1 or 2 at every r
    "targets":    ["qv", "qkv", "all"],   # names from LORA_TARGET_SETS
    "lr":         [3e-4, 1e-3],
}
LORA_TARGET_SETS = {                      # module-name suffixes inside each BERT encoder layer
    "qv":  ["query", "value"],
    "qkv": ["query", "key", "value"],
    "all": ["query", "key", "value", "attention.output.dense", "intermediate.dense", "output.dense"],  # every encoder Linear
}                                         # never plain "dense": it also matches pooler.dense, which is trained in full instead
LORA_DROPOUT      = 0.1
SEARCH_SEEDS      = [1]      # seeds per combination in stage 1; keep them different from SEEDS so stage 2 is a fresh measurement
SEARCH_MAX_TRIALS = None     # None = full grid (3*2*3*2 = 36 per model); an int = that many combinations sampled at random

# ---- QLoRA (CUDA + bitsandbytes only) ----
QLORA_BITS    = [4]          # 4 = QLoRA proper (NF4 + double quantization). Add 8 to also run LoRA on an 8-bit (LLM.int8) base
QLORA_HPARAMS = "from_lora"  # "from_lora": QLoRA uses LoRA's picked combination (isolates the effect of quantization)
                             # "search": QLoRA gets its own stage-1 search (doubles the search time per bit width)

# ---- Claim text fed to the model ----
CLAIM_MODE  = "fixed"      # "fixed": every row uses FIXED_CLAIM (as test.py / inference.py will); "file": the (translated) Target column
FIXED_CLAIM = "Climate change and global warming is a serious concern."

# ---- Splits (by tweet: all 7 language/variant rows of a tweet go to the same split) ----
TRAIN_FRACTION = 0.6       # trains the models
VAL_FRACTION   = 0.2       # picks the best epoch AND ranks the methods
                           # the remaining 0.2 is `test`: held out, shown only for the recommended method
TRAIN_SOURCE_ROWS    = None     # optional cap on training tweets (fast first pass); None = whole train split
TRAIN_VARIANT_POLICY = "all"    # "all": all 7 rows of each training tweet; "one_per_language": 1 random variant per language
SPLIT_SEED           = 2024

# ---- Training ----
SEEDS            = [42, 43]    # stage 2 seeds; 2 seeds show how noisy the comparison is; use [42] for a faster first pass
EPOCHS           = 3
BATCH_SIZE       = 16
EVAL_BATCH_SIZE  = 64
MAX_LEN          = 128         # checked against the token-length stats printed below
WEIGHT_DECAY     = 0.01        # not applied to bias / LayerNorm
WARMUP_FRACTION  = 0.1
GRAD_CLIP        = 1.0
HEAD_DROPOUT     = 0.1
PATIENCE         = 2
EARLY_STOP_MONITOR = "val"     # best epoch is chosen on this set
RANK_SET           = "val"     # methods are ranked on this set; `test` is never used to choose


## 2. Setup: install, platform, device

In [2]:
# ============================ 2. Install + imports + platform + device ============================
import os, sys, json, time, math, random, re, gc, glob, hashlib, platform, shutil, unicodedata, zipfile, subprocess, warnings
import traceback, datetime as dt
from pathlib import Path
from importlib import metadata as importlib_metadata

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")   # let unsupported ops fall back to CPU on Apple silicon
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")


def _pip(*pkgs):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *pkgs], capture_output=True, text=True)
    if r.returncode:
        print(f"[pip] failed for {pkgs}:\n{r.stderr[-800:]}")


if INSTALL_DEPS:
    _pip("transformers", "peft", "accelerate", "mlflow", "scikit-learn", "pandas", "torchao>=0.16.0")
    if platform.system() == "Linux":
        _pip("bitsandbytes")       # needed only for QLoRA (CUDA); skipped on Mac

import numpy as np
import pandas as pd
import torch

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/working")

# ---- Paths ----
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive") / DRIVE_FOLDER
    BASE_DIR.mkdir(parents=True, exist_ok=True)
    OUT_DIR = BASE_DIR / "experiment_outputs"
    WORK_DIR = Path("/content")
elif IN_KAGGLE:
    BASE_DIR = Path("/kaggle/working")
    OUT_DIR = BASE_DIR / "experiment_outputs"      # download this folder (or "Save Version") before the session ends
    WORK_DIR = BASE_DIR
else:
    BASE_DIR = Path.cwd()
    OUT_DIR = Path(LOCAL_OUT_DIR or "experiment_outputs").resolve()
    WORK_DIR = Path.cwd()
for sub in ("results", "predictions", "reports", "splits", "mlartifacts"):
    (OUT_DIR / sub).mkdir(parents=True, exist_ok=True)


def locate_data() -> Path:
    """Unzip experiment_data.zip (a fresh copy every session) and return the folder with semeval.csv."""
    if LOCAL_DATA_DIR and (Path(LOCAL_DATA_DIR) / "semeval.csv").exists():
        return Path(LOCAL_DATA_DIR)
    zips = []
    if IN_COLAB:
        zips.append(BASE_DIR / DATA_ZIP_NAME)                       # MyDrive/<DRIVE_FOLDER>/experiment_data.zip
    if IN_KAGGLE:
        for p in Path("/kaggle/input").rglob("semeval.csv"):        # Kaggle auto-extracts uploaded zips
            if (p.parent / "manifest.json").exists():
                return p.parent
        zips += list(Path("/kaggle/input").rglob(DATA_ZIP_NAME))
    zips += [Path(DATA_ZIP_NAME), Path("experiment") / DATA_ZIP_NAME]
    for z in zips:
        if z.exists():
            dest = WORK_DIR / "experiment_data_unzipped"
            shutil.rmtree(dest, ignore_errors=True)                  # never mix files from an older bundle
            with zipfile.ZipFile(z) as zf:
                zf.extractall(dest)
            if not (dest / "semeval.csv").exists():
                raise FileNotFoundError(f"{z} has no semeval.csv. Rebuild it with the current experiment/prepare_data.py.")
            print(f"Unzipped {z} -> {dest}")
            return dest
    for c in (Path("experiment_data"), Path("experiment/experiment_data")):
        if (c / "semeval.csv").exists():
            return c
    raise FileNotFoundError(
        f"Could not find the dataset. Looked for {DATA_ZIP_NAME} in: {[str(x) for x in zips]}. "
        f"Colab: upload it to MyDrive/{DRIVE_FOLDER}/. See experiment/SETUP.md.")


# ---- Device: Apple MPS first, then CUDA GPU, then CPU ----
def pick_device():
    mps = getattr(torch.backends, "mps", None)
    if mps is not None and mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


DEVICE = pick_device()
if DEVICE.type == "cpu" and (IN_COLAB or IN_KAGGLE) and not ALLOW_CPU_ON_CLOUD:
    raise RuntimeError("No GPU found. Colab: Runtime > Change runtime type > T4 GPU, then restart and run again. "
                       "Kaggle: Settings > Accelerator > GPU T4/P100. (Or set ALLOW_CPU_ON_CLOUD = True.)")
USE_AMP = DEVICE.type == "cuda"        # fp16 mixed precision on the T4; plain fp32 on MPS/CPU


def _ver(pkg):
    try:
        return importlib_metadata.version(pkg)
    except Exception:
        return None


ENV_INFO = dict(
    python=platform.python_version(), platform=platform.platform(),
    runtime="colab" if IN_COLAB else "kaggle" if IN_KAGGLE else "local",
    device=DEVICE.type,
    gpu=torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else None,
    gpu_mem_gb=round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1) if DEVICE.type == "cuda" else None,
    **{f"lib_{p}": _ver(p) for p in ("torch", "transformers", "peft", "bitsandbytes", "accelerate", "mlflow", "scikit-learn", "pandas")},
)
print(f"Device: {DEVICE}  (AMP fp16: {USE_AMP})   Outputs: {OUT_DIR}")
print({k: v for k, v in ENV_INFO.items() if v is not None})


/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda  (AMP fp16: True)   Outputs: /content/drive/MyDrive/EcoConsensus/experiment_outputs
{'python': '3.13.16', 'platform': 'Linux-6.6.159+-x86_64-with-glibc2.39', 'runtime': 'colab', 'device': 'cuda', 'gpu': 'Tesla T4', 'gpu_mem_gb': 14.6, 'lib_torch': '2.14.1+cu130', 'lib_transformers': '5.22.2', 'lib_peft': '0.21.2', 'lib_bitsandbytes': '0.50.2', 'lib_accelerate': '1.15.0', 'lib_mlflow': '3.18.0', 'lib_scikit-learn': '1.9.1', 'lib_pandas': '3.1.1'}


## 3. Helpers

In [3]:
# ============================ 3. Shared helpers: seeding, JSON, preprocessing, metrics ============================
LABELS = ["favour", "neutral", "against"]          # ids 0, 1, 2 (final project mapping, see docs/TRD.md)
L2I = {l: i for i, l in enumerate(LABELS)}
SPLIT_NAMES = ["val", "test"]                       # evaluation splits (train is separate)


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def stable_hash(obj, n=10) -> str:
    return hashlib.sha1(json.dumps(obj, sort_keys=True, default=str).encode()).hexdigest()[:n]


def clean(o):
    """Make an object JSON-safe: numpy -> python, NaN/inf -> None."""
    if isinstance(o, dict):
        return {str(k): clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [clean(v) for v in o]
    if isinstance(o, np.generic):
        o = o.item()
    if isinstance(o, float) and not math.isfinite(o):
        return None
    return o


def write_json(path: Path, obj):
    tmp = Path(str(path) + ".tmp")
    tmp.write_text(json.dumps(clean(obj), indent=2, ensure_ascii=False), encoding="utf-8")
    tmp.replace(path)


# ---- Preprocessing: one function, same rules as the TRD (emojis kept, @USER, URL, NFC, whitespace) ----
_URL = re.compile(r"https?://\S+|www\.\S+", re.I)
_HANDLE = re.compile(r"@\w+")
_SEMST = re.compile(r"#SemST\b", re.I)      # SemEval collection tag, not content
_HASH = re.compile(r"#(?=\S)")              # keep the hashtag word, drop '#'


def preprocess(text) -> str:
    if text is None:
        return ""
    t = unicodedata.normalize("NFC", str(text))
    t = _SEMST.sub(" ", t)
    t = _URL.sub("URL", t)
    t = _HANDLE.sub("@USER", t)
    t = _HASH.sub("", t)
    return re.sub(r"\s+", " ", t).strip()


# ---- Metrics (Macro-F1 first; labels fixed to the 3 classes so a missing class counts as 0) ----
def compute_metrics(y_true, y_pred, languages=None, variants=None) -> dict:
    from sklearn.metrics import (accuracy_score, cohen_kappa_score, confusion_matrix, f1_score,
                                 matthews_corrcoef, precision_recall_fscore_support)
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    labs = [0, 1, 2]
    p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=labs, zero_division=0)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        out = dict(
            n=int(len(y_true)),
            macro_f1=f1_score(y_true, y_pred, labels=labs, average="macro", zero_division=0),
            accuracy=accuracy_score(y_true, y_pred),
            weighted_f1=f1_score(y_true, y_pred, labels=labs, average="weighted", zero_division=0),
            mcc=matthews_corrcoef(y_true, y_pred),
            kappa=cohen_kappa_score(y_true, y_pred, labels=labs),
            per_class={LABELS[i]: dict(precision=p[i], recall=r[i], f1=f[i], support=int(s[i])) for i in labs},
            confusion=confusion_matrix(y_true, y_pred, labels=labs).tolist(),
        )
    if languages is not None:
        languages = np.asarray(languages)
        out["by_language"] = {lg: f1_score(y_true[languages == lg], y_pred[languages == lg], labels=labs,
                                           average="macro", zero_division=0)
                              for lg in sorted(set(languages))}
    if variants is not None:                          # "language/variant", e.g. "hindi/hinglish"
        variants = np.asarray(variants)
        out["by_variant"] = {v: f1_score(y_true[variants == v], y_pred[variants == v], labels=labs,
                                         average="macro", zero_division=0)
                             for v in sorted(set(variants))}
    return clean(out)


def split_metrics(df, pred) -> dict:
    """Metrics for one split with per-language and per-variant Macro-F1."""
    return compute_metrics(df["y"], pred, df["language"], df["language"] + "/" + df["variant"])


## 4. Data and frozen splits

In [4]:
# ============================ 4. Load data, preprocess, build frozen splits ============================
from sklearn.model_selection import train_test_split

DATA_DIR = locate_data()
MANIFEST = json.loads((DATA_DIR / "manifest.json").read_text(encoding="utf-8"))
RAW = pd.read_csv(DATA_DIR / "semeval.csv", dtype=str, keep_default_na=False)
print(f"Loaded semeval.csv: {len(RAW)} rows, {RAW['source_id'].nunique()} tweets | data hash: {MANIFEST['data_hash']}")
print(pd.crosstab([RAW["language"], RAW["variant"]], RAW["label"]).reindex(columns=LABELS, fill_value=0).to_string())
for note in MANIFEST.get("notes", []):
    print("Note:", note)


def prep(df: pd.DataFrame):
    """Apply the shared preprocessing, set the claim, drop empty comments, add integer label ids."""
    df = df.copy()
    df["claim"] = FIXED_CLAIM if CLAIM_MODE == "fixed" else df["claim"]
    df["claim"] = df["claim"].map(preprocess)
    df["comment"] = df["comment"].map(preprocess)
    before = len(df)
    df = df[df["comment"] != ""].reset_index(drop=True)
    bad = set(df["label"]) - set(LABELS)
    assert not bad, f"unexpected labels {bad}"
    df["y"] = df["label"].map(L2I).astype(int)
    return df, before - len(df)


def _report_table(splits: dict) -> str:
    rows = []
    for name, df in splits.items():
        ct = pd.crosstab(df["language"], df["label"]).reindex(columns=LABELS, fill_value=0)
        for lang, r in ct.iterrows():
            rows.append(f"| {name} | {lang} | {int(r.sum())} | " + " | ".join(str(int(r[l])) for l in LABELS) + " |")
        tot = df["label"].value_counts().reindex(LABELS, fill_value=0)
        rows.append(f"| **{name}** | **all ({df['source_id'].nunique()} tweets)** | **{len(df)}** | "
                    + " | ".join(f"**{int(tot[l])}**" for l in LABELS) + " |")
    return "| split | language | rows | " + " | ".join(LABELS) + " |\n|---|---|---|" + "---|" * len(LABELS) + "\n" + "\n".join(rows)


def make_splits():
    assert CLAIM_MODE in ("fixed", "file"), "CLAIM_MODE must be 'fixed' or 'file'"
    assert 0 < TRAIN_FRACTION and 0 < VAL_FRACTION and TRAIN_FRACTION + VAL_FRACTION < 1, "fractions must leave room for test"
    key = stable_hash(dict(data=MANIFEST["data_hash"], train=TRAIN_FRACTION, val=VAL_FRACTION, n=TRAIN_SOURCE_ROWS,
                           policy=TRAIN_VARIANT_POLICY, claim=CLAIM_MODE, fixed=FIXED_CLAIM, seed=SPLIT_SEED))
    sdir = OUT_DIR / "splits" / key
    names = ["train", "val", "test"]
    if all((sdir / f"{n}.csv").exists() for n in names):
        splits = {n: pd.read_csv(sdir / f"{n}.csv", dtype={"source_id": str, "label": str, "claim": str, "comment": str,
                                                          "language": str, "variant": str}, keep_default_na=False)
                  for n in names}
        dropped = json.loads((sdir / "dropped.json").read_text()) if (sdir / "dropped.json").exists() else {}
        print(f"Re-using frozen splits {key}")
        return key, splits, dropped

    sem, d = prep(RAW)
    dropped = dict(semeval=d)

    # --- split by SOURCE tweet, stratified on its label, so all languages/variants of a tweet stay together ---
    src = sem.drop_duplicates("source_id")[["source_id", "label"]]
    tr_src, rest = train_test_split(src, train_size=TRAIN_FRACTION, stratify=src["label"], random_state=SPLIT_SEED)
    va_src, te_src = train_test_split(rest, train_size=VAL_FRACTION / (1 - TRAIN_FRACTION), stratify=rest["label"],
                                      random_state=SPLIT_SEED)
    if TRAIN_SOURCE_ROWS and TRAIN_SOURCE_ROWS < len(tr_src):
        tr_src, _ = train_test_split(tr_src, train_size=TRAIN_SOURCE_ROWS, stratify=tr_src["label"], random_state=SPLIT_SEED)

    train = sem[sem["source_id"].isin(set(tr_src["source_id"]))]
    if TRAIN_VARIANT_POLICY == "one_per_language":
        train = train.groupby(["source_id", "language"], group_keys=False).sample(1, random_state=SPLIT_SEED)
    elif TRAIN_VARIANT_POLICY != "all":
        raise ValueError("TRAIN_VARIANT_POLICY must be 'all' or 'one_per_language'")
    splits = dict(
        train=train.sort_values(["source_id", "language", "variant"]).reset_index(drop=True),
        val=sem[sem["source_id"].isin(set(va_src["source_id"]))].reset_index(drop=True),
        test=sem[sem["source_id"].isin(set(te_src["source_id"]))].reset_index(drop=True),
    )

    # --- leakage check: no tweet in two splits ---
    ids = {n: set(df["source_id"]) for n, df in splits.items()}
    for i, a in enumerate(names):
        for b in names[i + 1:]:
            assert not (ids[a] & ids[b]), f"tweets shared between {a} and {b}"
    sdir.mkdir(parents=True, exist_ok=True)
    for n, df in splits.items():
        df.to_csv(sdir / f"{n}.csv", index=False, encoding="utf-8")
    write_json(sdir / "dropped.json", dropped)
    (sdir / "split_report.md").write_text(
        f"# Split report ({key})\n\nSemEval subset, split by tweet. Seed {SPLIT_SEED}. Fractions train {TRAIN_FRACTION} / "
        f"val {VAL_FRACTION} / test {1 - TRAIN_FRACTION - VAL_FRACTION:.2f}. Training cap {TRAIN_SOURCE_ROWS or 'none'}, "
        f"variant policy `{TRAIN_VARIANT_POLICY}`. Claim mode `{CLAIM_MODE}`.\n"
        f"Rows dropped for empty comments: {dropped}\n\n" + _report_table(splits) + "\n", encoding="utf-8")
    print(f"Created frozen splits {key}")
    return key, splits, dropped


SPLIT_KEY, SPLITS, DROPPED = make_splits()
print((OUT_DIR / "splits" / SPLIT_KEY / "split_report.md").read_text(encoding="utf-8"))


Unzipped /content/drive/MyDrive/EcoConsensus/experiment_data.zip -> /content/experiment_data_unzipped
Loaded semeval.csv: 2100 rows, 300 tweets | data hash: a217aa4a0c41c49c
label                      favour  neutral  against
language variant                                   
bengali  banglish             171      103       26
         bengali_code_mix     171      103       26
         pure_bengali         171      103       26
english  original             171      103       26
hindi    hindi_code_mix       171      103       26
         hinglish             171      103       26
         pure_hindi           171      103       26
Note: semeval: 300 of 564 tweets, 7 language/variant groups: bengali/banglish, bengali/bengali_code_mix, bengali/pure_bengali, english/original, hindi/hindi_code_mix, hindi/hinglish, hindi/pure_hindi
Created frozen splits 1af4afea20
# Split report (1af4afea20)

SemEval subset, split by tweet. Seed 2024. Fractions train 0.6 / val 0.2 / test 0.20. Training c

## 5. Tokenizer check

In [5]:
# ============================ 5. Tokenizer check: downloads each model's tokenizer, prints length + [UNK] stats ============================
from transformers import AutoTokenizer

TOKENIZERS, TOKEN_STATS, UNAVAILABLE = {}, {}, {}
_all = pd.concat(SPLITS.values(), ignore_index=True)

for key, name in MODELS.items():
    try:
        tok = AutoTokenizer.from_pretrained(name)
        enc = tok(_all["claim"].tolist(), _all["comment"].tolist(), truncation=False, verbose=False)
        lens = np.array([len(x) for x in enc["input_ids"]])
        unk_id = tok.unk_token_id
        unk = {}
        if unk_id is not None:
            for lg in sorted(_all["language"].unique()):
                ids = [x for x, l in zip(enc["input_ids"], _all["language"]) if l == lg]
                total = sum(len(x) for x in ids)
                unk[lg] = round(sum(x.count(unk_id) for x in ids) / max(1, total), 5)
        TOKENIZERS[key] = tok
        TOKEN_STATS[key] = dict(model=name, p50=int(np.percentile(lens, 50)), p95=int(np.percentile(lens, 95)),
                                p99=int(np.percentile(lens, 99)), max=int(lens.max()),
                                truncated_pct_at_max_len=round(float((lens > MAX_LEN).mean() * 100), 2),
                                unk_rate_by_language=unk)
        print(f"{key:10s} {TOKEN_STATS[key]}")
        for lg in sorted(_all["language"].unique()):
            ex = _all[_all["language"] == lg].iloc[0]
            print(f"   {lg:8s} sample tokens:", tok.tokenize(ex["comment"])[:12])
    except Exception as e:                      # keep going with the other model
        UNAVAILABLE[key] = f"tokenizer failed: {type(e).__name__}: {str(e)[:300]}"
        print(f"[!] {key}: {UNAVAILABLE[key]}")

if TOKEN_STATS and max(s["p95"] for s in TOKEN_STATS.values()) > MAX_LEN:
    print(f"[!] 95th percentile length exceeds MAX_LEN={MAX_LEN}; consider raising it (costs memory/time).")
if not TOKENIZERS:
    raise RuntimeError("No tokenizer could be loaded; see messages above.")


muril      {'model': 'google/muril-base-cased', 'p50': 39, 'p95': 51, 'p99': 58, 'max': 70, 'truncated_pct_at_max_len': 0.0, 'unk_rate_by_language': {'bengali': 0.0, 'english': 0.0, 'hindi': 0.0}}
   bengali  sample tokens: ['Am', '##ra', 'eta', 'deny', 'ko', '##rte', 'pari', 'na', ',', 'eta', 'so', '##tti']
   english  sample tokens: ['We', 'cant', 'deny', 'it', ',', 'its', 'really', 'happening', '.']
   hindi    sample tokens: ['हम', 'इससे', 'deny', 'नहीं', 'कर', 'सकते', ',', 'यह', 'सचमुच', 'हो', 'रहा', 'है']
indicbert  {'model': 'ai4bharat/IndicBERTv2-MLM-only', 'p50': 39, 'p95': 51, 'p99': 58, 'max': 66, 'truncated_pct_at_max_len': 0.0, 'unk_rate_by_language': {'bengali': 0.0, 'english': 0.0, 'hindi': 0.0}}
   bengali  sample tokens: ['Amr', '##a', 'et', '##a', 'deny', 'kort', '##e', 'pari', 'na', ',', 'et', '##a']
   english  sample tokens: ['We', 'cant', 'deny', 'it', ',', 'its', 'really', 'happening', '.']
   hindi    sample tokens: ['हम', 'इससे', 'deny', 'नहीं', 'कर', 'सकते', '

## 6. Training code

In [6]:
# ============================ 6. Training code: datasets, model builders (all methods), train loop ============================
import itertools
from contextlib import nullcontext
from transformers import AutoModelForSequenceClassification


# ---- Methods. LoRA-type specs carry `hp` = dict(r, alpha_mult, targets, lr), picked by the stage-1 search in cell 8 ----
def lora_method_ids():
    """(method id, kind, bits) for every LoRA-type method in RUN_METHODS."""
    out = [("lora", "lora", None)] if "lora" in RUN_METHODS else []
    if "qlora" in RUN_METHODS:
        for b in QLORA_BITS:
            assert b in (4, 8), "QLORA_BITS may only contain 4 and 8"
            out.append((f"qlora_{b}bit", "qlora", b))
    return out


def method_ids():
    ids = ["head_only"] if "head_only" in RUN_METHODS else []
    ids += [f"last{n}" for n in LAST_N_LIST] if "last_n" in RUN_METHODS else []
    return ids + [m for m, _, _ in lora_method_ids()]


def method_specs(chosen=None, model_key=None):
    """Stage-2 specs. chosen[(model_key, method_id)] is the hp picked in stage 1 (None if no search trial finished)."""
    specs = []
    if "head_only" in RUN_METHODS:
        specs.append(dict(id="head_only", kind="head_only", n_last=None))
    if "last_n" in RUN_METHODS:
        specs += [dict(id=f"last{n}", kind="last_n", n_last=n) for n in LAST_N_LIST]
    for mid, kind, bits in lora_method_ids():
        specs.append(dict(id=mid, kind=kind, n_last=None, bits=bits, hp=(chosen or {}).get((model_key, mid))))
    return specs


def lora_grid():
    """All combinations of LORA_SEARCH (or a fixed random sample of SEARCH_MAX_TRIALS of them)."""
    unknown = set(LORA_SEARCH["targets"]) - set(LORA_TARGET_SETS)
    assert not unknown, f"LORA_SEARCH targets {unknown} are not in LORA_TARGET_SETS"
    keys = ["r", "alpha_mult", "targets", "lr"]
    combos = [dict(zip(keys, v)) for v in itertools.product(*(LORA_SEARCH[k] for k in keys))]
    if SEARCH_MAX_TRIALS and SEARCH_MAX_TRIALS < len(combos):
        combos = random.Random(SPLIT_SEED).sample(combos, SEARCH_MAX_TRIALS)
    return combos


def hp_tag(hp):
    return f"r{hp['r']}_a{hp['r'] * hp['alpha_mult']}_{hp['targets']}_lr{hp['lr']:g}" if hp else "-"


def spec_lr(spec):
    return {"lr": spec["hp"]["lr"]} if spec.get("hp") else LR.get(spec["kind"], {})


# ---- Data: pre-tokenised sentence pairs (claim, comment), padded per batch ----
class PairDataset(torch.utils.data.Dataset):
    def __init__(self, df, tok, max_len):
        enc = tok(df["claim"].tolist(), df["comment"].tolist(), truncation=True, max_length=max_len)
        self.keys = [k for k in ("input_ids", "attention_mask", "token_type_ids") if k in enc]
        self.enc = {k: enc[k] for k in self.keys}
        self.y = df["y"].to_numpy()
        self.lengths = [len(x) for x in self.enc["input_ids"]]

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        item = {k: self.enc[k][i] for k in self.keys}
        item["labels"] = int(self.y[i])
        return item


def make_collate(pad_id):
    def collate(batch):
        width = max(len(b["input_ids"]) for b in batch)
        out = {}
        for k in batch[0]:
            if k == "labels":
                out[k] = torch.tensor([b[k] for b in batch], dtype=torch.long)
            else:
                fill = pad_id if k == "input_ids" else 0
                out[k] = torch.tensor([b[k] + [fill] * (width - len(b[k])) for b in batch], dtype=torch.long)
        return out
    return collate


_DS_CACHE = {}


def get_datasets(model_key):
    if model_key not in _DS_CACHE:
        tok = TOKENIZERS[model_key]
        _DS_CACHE[model_key] = {n: PairDataset(df, tok, MAX_LEN) for n, df in SPLITS.items()}
    return _DS_CACHE[model_key]


def amp_ctx():
    return torch.autocast(device_type="cuda", dtype=torch.float16) if USE_AMP else nullcontext()


@torch.no_grad()
def predict_probs(model, ds, pad_id):
    model.eval()
    collate = make_collate(pad_id)
    order = np.argsort(ds.lengths)                    # length-sorted batches = less padding = faster
    probs = np.zeros((len(ds), 3), dtype=np.float32)
    for s in range(0, len(order), EVAL_BATCH_SIZE):
        idx = order[s:s + EVAL_BATCH_SIZE]
        batch = collate([ds[int(i)] for i in idx])
        batch.pop("labels")
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        with amp_ctx():
            logits = model(**batch).logits
        probs[idx] = torch.softmax(logits.float(), dim=-1).cpu().numpy()
    return probs


# ---- Models and methods ----
def _is_head(name):                       # classifier + pooler (the pooler may be newly initialised in MLM-only checkpoints)
    return any(t in name for t in ("classifier", "pooler", "score"))


def count_params(model):
    trainable = total = 0
    for p in model.parameters():
        n = p.numel() * (2 if p.__class__.__name__ == "Params4bit" else 1)  # 4-bit weights are packed 2 per byte
        total += n
        trainable += n if p.requires_grad else 0
    return trainable, total


def build_model(model_name, spec):
    kw = dict(num_labels=3, id2label=dict(enumerate(LABELS)), label2id=L2I,
              ignore_mismatched_sizes=True)   # a checkpoint that already has a head of another size gets a fresh 3-class head
    if DEVICE.type == "mps":
        kw["attn_implementation"] = "eager"      # PyTorch's fused attention on MPS does not support dropout in training
    kind = spec["kind"]

    def load(**extra):
        try:
            return AutoModelForSequenceClassification.from_pretrained(model_name, classifier_dropout=HEAD_DROPOUT, **kw, **extra)
        except TypeError:                 # model class without classifier_dropout
            return AutoModelForSequenceClassification.from_pretrained(model_name, **kw, **extra)
    if kind == "qlora":
        from transformers import BitsAndBytesConfig
        from peft import prepare_model_for_kbit_training
        # The new classifier and the pooler stay in full precision: both are trained in full (modules_to_save below).
        # A quantized pooler inside modules_to_save is what broke QLoRA before (bitsandbytes AssertionError).
        # "pooler.dense", not "pooler": transformers 5 matches these names as a suffix of the full module name.
        skip = ["classifier", "pooler.dense"]
        if spec["bits"] == 4:
            qcfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
                                      bnb_4bit_compute_dtype=torch.float16, llm_int8_skip_modules=skip)
        else:
            qcfg = BitsAndBytesConfig(load_in_8bit=True, llm_int8_skip_modules=skip)
        model = load(quantization_config=qcfg, device_map={"": 0})
        quantized_head = [n for n, m in model.named_modules()
                          if _is_head(n) and type(m).__name__ in ("Linear4bit", "Linear8bitLt")]
        if quantized_head:
            raise RuntimeError(f"Head layers were quantized: {quantized_head}. They must stay full precision "
                               f"(check llm_int8_skip_modules for this transformers version).")
        # Casts the remaining full-precision weights to fp32 and freezes everything. Do not cast parameters by name
        # afterwards: a filter such as "pooler"/"bias" can hit packed quantized weights and corrupt them.
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=False)
    else:
        model = load()
    if kind in ("head_only", "last_n"):
        n_layers = model.config.num_hidden_layers
        for n, p in model.named_parameters():
            train = _is_head(n)
            if kind == "last_n":
                m = re.search(r"\.layer\.(\d+)\.", n)
                if m and int(m.group(1)) >= n_layers - spec["n_last"]:
                    train = True
            p.requires_grad = train
        if kind == "last_n" and not any(re.search(r"\.layer\.\d+\.", n) and p.requires_grad for n, p in model.named_parameters()):
            raise ValueError("Could not find encoder layers named '.layer.<i>.' - this architecture needs a custom rule.")
    else:                                   # lora / qlora
        from peft import LoraConfig, TaskType, get_peft_model
        hp = spec["hp"]
        head = ["classifier"] + (["pooler"] if getattr(model.base_model, "pooler", None) is not None else [])
        cfg = LoraConfig(task_type=TaskType.SEQ_CLS, r=hp["r"], lora_alpha=hp["r"] * hp["alpha_mult"],
                         lora_dropout=LORA_DROPOUT, target_modules=LORA_TARGET_SETS[hp["targets"]], modules_to_save=head)
        model = get_peft_model(model, cfg)
    if kind != "qlora":                     # qlora is already on the GPU (device_map)
        model.to(DEVICE)
    if sum(p.numel() for p in model.parameters() if p.requires_grad) == 0:
        raise ValueError("No trainable parameters.")
    return model


def make_optimizer(model, spec):
    lrs = spec_lr(spec)
    groups = {}
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        lr = lrs.get("head_lr", lrs["lr"]) if _is_head(n) else lrs["lr"]
        no_decay = n.endswith(".bias") or "LayerNorm" in n or "layer_norm" in n.lower()
        groups.setdefault((lr, 0.0 if no_decay else WEIGHT_DECAY), []).append(p)
    return torch.optim.AdamW([dict(params=ps, lr=lr, weight_decay=wd) for (lr, wd), ps in groups.items()])


def free_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if DEVICE.type == "mps":
        torch.mps.empty_cache()


def peak_mem_mb():
    if DEVICE.type == "cuda":
        return round(torch.cuda.max_memory_allocated() / 2**20, 1)
    if DEVICE.type == "mps":
        return round(torch.mps.current_allocated_memory() / 2**20, 1)
    return None


def train_one(model_key, spec, seed, micro_bs, accum):
    """Train one (model, method, seed). Returns a result dict (without status fields)."""
    t_start = time.perf_counter()
    set_seed(seed)
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    tok, name = TOKENIZERS[model_key], MODELS[model_key]
    ds = get_datasets(model_key)
    pad_id = tok.pad_token_id if tok.pad_token_id is not None else 0
    model = build_model(name, spec)
    trainable, total = count_params(model)

    y = SPLITS["train"]["y"].to_numpy()
    counts = np.bincount(y, minlength=3).astype(float)
    cw = np.where(counts > 0, len(y) / (3.0 * np.maximum(counts, 1)), 1.0)  # class-weighted loss
    loss_fn = torch.nn.CrossEntropyLoss(weight=torch.tensor(cw, dtype=torch.float32, device=DEVICE))

    loader = torch.utils.data.DataLoader(ds["train"], batch_size=micro_bs, shuffle=True, collate_fn=make_collate(pad_id),
                                         generator=torch.Generator().manual_seed(seed))
    steps_per_epoch = math.ceil(len(loader) / accum)
    total_steps = steps_per_epoch * EPOCHS
    warm = max(1, int(WARMUP_FRACTION * total_steps))
    opt = make_optimizer(model, spec)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: s / warm if s < warm else max(0.0, (total_steps - s) / max(1, total_steps - warm)))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    params = [p for p in model.parameters() if p.requires_grad]

    history, best, best_state, since_best, train_sec = [], -1.0, None, 0, 0.0
    for epoch in range(1, EPOCHS + 1):
        model.train()
        t0, loss_sum, n_batches = time.perf_counter(), 0.0, 0
        opt.zero_grad(set_to_none=True)
        for i, batch in enumerate(loader, 1):
            labels = batch.pop("labels").to(DEVICE)
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            with amp_ctx():
                logits = model(**batch).logits
            loss = loss_fn(logits.float(), labels)
            scaler.scale(loss / accum).backward()
            loss_sum += loss.item()
            n_batches += 1
            if i % accum == 0 or i == len(loader):
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(params, GRAD_CLIP)
                scaler.step(opt)
                scaler.update()
                sched.step()
                opt.zero_grad(set_to_none=True)
        ep_sec = time.perf_counter() - t0
        train_sec += ep_sec

        row = dict(epoch=epoch, train_loss=loss_sum / max(1, n_batches), epoch_train_sec=round(ep_sec, 1))
        for split in ("val",):
            pr = predict_probs(model, ds[split], pad_id)
            m = compute_metrics(SPLITS[split]["y"], pr.argmax(1))
            row[f"{split}_macro_f1"], row[f"{split}_accuracy"] = m["macro_f1"], m["accuracy"]
        history.append(row)
        score = row[f"{EARLY_STOP_MONITOR}_macro_f1"]
        print(f"    epoch {epoch}: loss {row['train_loss']:.3f} | val F1 {row['val_macro_f1']:.3f} | "
              f"val acc {row['val_accuracy']:.3f} | {ep_sec:.0f}s")
        if score > best:
            best, since_best = score, 0
            best_state = {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}
        else:
            since_best += 1
            if since_best >= PATIENCE:
                print("    early stopping")
                break

    with torch.no_grad():                    # restore the best epoch
        for n, p in model.named_parameters():
            if n in best_state:
                p.copy_(best_state[n].to(p.device))
    best_epoch = int(np.argmax([h[f"{EARLY_STOP_MONITOR}_macro_f1"] for h in history]) + 1)

    metrics, pred_frames = {}, []
    for split in SPLIT_NAMES:
        pr = predict_probs(model, ds[split], pad_id)
        df = SPLITS[split]
        metrics[split] = split_metrics(df, pr.argmax(1))
        pf = df[["source_id", "language", "variant", "label"]].copy()
        pf.insert(0, "split", split)
        pf["pred"] = [LABELS[i] for i in pr.argmax(1)]
        for j, l in enumerate(LABELS):
            pf[f"p_{l}"] = pr[:, j].round(5)
        pred_frames.append(pf)

    result = dict(trainable_params=trainable, total_params=total, trainable_pct=100.0 * trainable / total,
                  train_time_sec=round(train_sec, 1), total_time_sec=round(time.perf_counter() - t_start, 1),
                  peak_mem_mb=peak_mem_mb(), best_epoch=best_epoch, epochs_run=len(history), history=history,
                  metrics=metrics, micro_batch=micro_bs, grad_accum=accum)
    del model, opt, sched, scaler, best_state, params
    return result, pd.concat(pred_frames, ignore_index=True)


## 7. Reference baselines

In [7]:
# ============================ 7. Reference baselines on the same splits (Majority, TF-IDF + Logistic Regression) ============================
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import FeatureUnion, make_pipeline

RESULTS_DIR = OUT_DIR / "results"


def _pair_text(df):
    return (df["claim"] + " [SEP] " + df["comment"]).tolist()


def run_baselines(config_hash_base):
    """Fit both baselines on the training split; C is chosen on val only. Returns list of result dicts."""
    train, out = SPLITS["train"], []
    maj = int(train["y"].value_counts().idxmax())
    out.append(dict(run_name="baseline__majority", method_id="majority", params=dict(label=LABELS[maj]),
                    metrics={s: split_metrics(SPLITS[s], np.full(len(SPLITS[s]), maj)) for s in SPLIT_NAMES}))

    token_re = r"(?u)[\wऀ-ॿঀ-৿]+"            # keeps Devanagari/Bengali words intact (matras are not \w)
    best_c, best_f1, best_model = None, -1, None
    for C in (0.3, 1, 3, 10):
        feats = FeatureUnion([
            ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True, lowercase=True)),
            ("word", TfidfVectorizer(analyzer="word", ngram_range=(1, 2), token_pattern=token_re, sublinear_tf=True, lowercase=True)),
        ])
        pipe = make_pipeline(feats, LogisticRegression(C=C, class_weight="balanced", max_iter=3000))
        pipe.fit(_pair_text(train), train["y"])
        f1 = compute_metrics(SPLITS["val"]["y"], pipe.predict(_pair_text(SPLITS["val"])))["macro_f1"]
        if f1 > best_f1:
            best_c, best_f1, best_model = C, f1, pipe
    out.append(dict(run_name="baseline__tfidf_lr", method_id="tfidf_lr", params=dict(C=best_c, chosen_on="val"),
                    metrics={s: split_metrics(SPLITS[s], best_model.predict(_pair_text(SPLITS[s]))) for s in SPLIT_NAMES}))
    for r in out:
        r.update(kind="baseline", config_hash=config_hash_base, status="ok", seed=None, model_key="-",
                 finished_at=dt.datetime.now().isoformat(timespec="seconds"))
    return out


## 8. Run the experiments

In [8]:
# ============================ 8. Run everything (serial, resumable) ============================
# Stage 1 "search": every LORA_SEARCH combination, seeds SEARCH_SEEDS, scored on val -> one picked combination per model.
# Stage 2 "compare": every method with all SEEDS; LoRA / QLoRA use the stage-1 pick for that model.
# A run is skipped if a finished result with the same settings already exists in <outputs>/results.
# If Colab disconnects: reconnect and Run all again - it continues where it stopped.

COMMON = dict(epochs=EPOCHS, bs=BATCH_SIZE, max_len=MAX_LEN, wd=WEIGHT_DECAY, warmup=WARMUP_FRACTION, clip=GRAD_CLIP,
              dropout=HEAD_DROPOUT, patience=PATIENCE, monitor=EARLY_STOP_MONITOR, lora_dropout=LORA_DROPOUT,
              data=MANIFEST["data_hash"], split=SPLIT_KEY, claim_mode=CLAIM_MODE, amp=USE_AMP)
BASE_HASH = stable_hash(dict(kind="baseline", data=MANIFEST["data_hash"], split=SPLIT_KEY))


def run_hash(mk, spec, seed):
    extra = dict(target_modules=LORA_TARGET_SETS[spec["hp"]["targets"]]) if spec.get("hp") else {}
    return stable_hash(dict(COMMON, model=MODELS[mk], spec=spec, seed=seed, lr=spec_lr(spec), **extra))


def result_path(run_name, h):
    return RESULTS_DIR / f"{run_name}__{h}.json"


def load_ok(run):
    p = result_path(run["run_name"], run["config_hash"])
    if p.exists():
        r = json.loads(p.read_text(encoding="utf-8"))
        if r.get("status") == "ok":
            return r
    return None


def searched_methods():
    """LoRA-type methods that get a stage-1 search. QLoRA reuses LoRA's pick unless QLORA_HPARAMS == "search"."""
    lm = lora_method_ids()
    has_lora = any(m == "lora" for m, _, _ in lm)
    return [x for x in lm if x[0] == "lora" or QLORA_HPARAMS == "search" or not has_lora]


def plan_search():
    runs = []
    for mk in MODELS:
        for mid, kind, bits in searched_methods():
            for hp in lora_grid():
                for seed in SEARCH_SEEDS:
                    spec = dict(id=mid, kind=kind, n_last=None, bits=bits, hp=hp)
                    runs.append(dict(run_name=f"{mk}__search__{mid}__{hp_tag(hp)}__seed{seed}", stage="search",
                                     model_key=mk, spec=spec, seed=seed, config_hash=run_hash(mk, spec, seed)))
    return runs


def choose_hp(search_runs):
    """Per (model, method): mean val Macro-F1 per combination; take the best, then the fewest trainable parameters
    among combinations within max(1 std of the best, 0.010) of it (same parsimony rule as the method ranking)."""
    by = {}
    for run in search_runs:
        r = load_ok(run)
        if r:
            by.setdefault((run["model_key"], run["spec"]["id"]), {}).setdefault(hp_tag(run["spec"]["hp"]), []).append(r)
    chosen, table = {}, {}
    for key, combos in by.items():
        rows = []
        for tag, rs in combos.items():
            f1 = [r["metrics"]["val"]["macro_f1"] for r in rs]
            rows.append(dict(tag=tag, hp=rs[0]["hp"], mean=float(np.mean(f1)),
                             std=float(np.std(f1, ddof=1)) if len(f1) > 1 else 0.0, n=len(rs),
                             params=int(rs[0]["trainable_params"]),
                             against=float(np.mean([r["metrics"]["val"]["per_class"]["against"]["f1"] for r in rs]))))
        best = max(rows, key=lambda x: x["mean"])
        tol = max(best["std"], 0.01)
        pick = min((x for x in rows if x["mean"] >= best["mean"] - tol), key=lambda x: (x["params"], -x["mean"]))
        chosen[key] = pick["hp"]
        table[key] = dict(rows=sorted(rows, key=lambda x: -x["mean"]), best=best["tag"], pick=pick["tag"], tol=tol)
    for mk in MODELS:                                   # QLoRA without its own search takes LoRA's pick
        for mid, _, _ in lora_method_ids():
            if (mk, mid) not in chosen and mid not in {m for m, _, _ in searched_methods()} and (mk, "lora") in chosen:
                chosen[(mk, mid)] = chosen[(mk, "lora")]
    return chosen, table


def plan_runs(chosen):
    runs = []
    for seed in SEEDS:                                   # all methods for seed 1 finish before seed 2 starts
        for mk in MODELS:
            for spec in method_specs(chosen, mk):
                runs.append(dict(run_name=f"{mk}__{spec['id']}__seed{seed}", stage="compare", model_key=mk, spec=spec,
                                 seed=seed, config_hash=run_hash(mk, spec, seed)))
    return runs


SEARCH_PLAN = plan_search()
PLANNED, CHOSEN_HP, SEARCH_TABLE = [], {}, {}
ACTIVE_HASHES = {r["config_hash"] for r in SEARCH_PLAN} | {BASE_HASH}


def qlora_blocker():
    if DEVICE.type != "cuda":
        return f"QLoRA needs a CUDA GPU (device is {DEVICE.type})"
    try:
        import bitsandbytes  # noqa: F401
    except Exception as e:
        return f"bitsandbytes not usable: {type(e).__name__}: {str(e)[:200]}"
    return None


# ---- MLflow (optional; failures never stop the experiment) ----
# Store = SQLite file <outputs>/mlflow.db, artifacts in <outputs>/mlartifacts (both on Drive in Colab).
# SQLite on the Drive mount can hit file-locking errors, so Colab writes a local copy in /content and
# copies it to Drive after every run (sync_mlflow). After a disconnect the Drive copy is restored first.
MLFLOW_OK = False
MLFLOW_DB_OUT = OUT_DIR / "mlflow.db"
MLFLOW_DB = (WORK_DIR / "mlflow.db") if IN_COLAB else MLFLOW_DB_OUT


def sync_mlflow():
    if MLFLOW_OK and MLFLOW_DB != MLFLOW_DB_OUT and MLFLOW_DB.exists():
        try:
            shutil.copy2(MLFLOW_DB, MLFLOW_DB_OUT)
        except Exception as e:
            print(f"    [mlflow sync failed] {type(e).__name__}: {str(e)[:150]}")


if USE_MLFLOW:
    try:
        import mlflow
        if MLFLOW_DB != MLFLOW_DB_OUT and MLFLOW_DB_OUT.exists() and not MLFLOW_DB.exists():
            shutil.copy2(MLFLOW_DB_OUT, MLFLOW_DB)            # continue the store saved by an earlier session
        mlflow.set_tracking_uri(f"sqlite:///{MLFLOW_DB}")
        if mlflow.get_experiment_by_name(EXPERIMENT_NAME) is None:
            mlflow.create_experiment(EXPERIMENT_NAME, artifact_location=(OUT_DIR / "mlartifacts").as_uri())
        mlflow.set_experiment(EXPERIMENT_NAME)
        MLFLOW_OK = True
        print(f"MLflow store: {MLFLOW_DB_OUT}  (artifacts: {OUT_DIR / 'mlartifacts'})")
    except Exception as e:
        print(f"[mlflow disabled] {type(e).__name__}: {e}")


def log_mlflow(rec):
    if not MLFLOW_OK:
        return
    try:
        with mlflow.start_run(run_name=rec["run_name"]):
            mlflow.log_params({k: rec.get(k) for k in ("model_key", "method_id", "seed", "config_hash", "stage") if rec.get(k) is not None})
            if rec.get("hp"):
                mlflow.log_params({f"hp_{k}": v for k, v in rec["hp"].items()})
            mlflow.log_params({f"cfg_{k}": str(v)[:250] for k, v in COMMON.items()})
            mlflow.set_tag("status", rec["status"])
            if rec.get("reason"):
                mlflow.set_tag("reason", str(rec["reason"])[:500])
            for h in rec.get("history", []):
                for k, v in h.items():
                    if k != "epoch" and v is not None:
                        mlflow.log_metric(k, v, step=h["epoch"])
            for split, m in (rec.get("metrics") or {}).items():
                if split == "test":
                    continue                              # held-out split is not logged per run; it appears only in the report
                mlflow.log_metrics({f"{split}_{k}": m[k] for k in ("macro_f1", "accuracy", "weighted_f1", "mcc", "kappa") if m.get(k) is not None})
                mlflow.log_metrics({f"{split}_f1_{c}": m["per_class"][c]["f1"] for c in LABELS})
                mlflow.log_metrics({f"{split}_f1_{re.sub(r'[^0-9A-Za-z_]', '_', v)}": f for v, f in (m.get("by_variant") or {}).items()})
            for k in ("trainable_params", "trainable_pct", "train_time_sec", "peak_mem_mb", "best_epoch"):
                if rec.get(k) is not None:
                    mlflow.log_metric(k, rec[k])
            mlflow.log_artifact(str(result_path(rec["run_name"], rec["config_hash"])))
    except Exception as e:
        print(f"    [mlflow log failed] {type(e).__name__}: {str(e)[:150]}")
    sync_mlflow()


def execute(run):
    """Run one transformer experiment with OOM fallback; always returns a record (ok / failed / skipped)."""
    spec = run["spec"]
    rec = dict(run_name=run["run_name"], kind="search" if run["stage"] == "search" else "transformer", stage=run["stage"],
               model_key=run["model_key"], model_name=MODELS[run["model_key"]], method_id=spec["id"], method_kind=spec["kind"],
               n_last=spec["n_last"], bits=spec.get("bits"), hp=spec.get("hp"), seed=run["seed"], config_hash=run["config_hash"],
               params=dict(lr=spec_lr(spec), epochs=EPOCHS, batch_size=BATCH_SIZE, max_len=MAX_LEN))
    if run["model_key"] in UNAVAILABLE:
        return dict(rec, status="skipped", reason=UNAVAILABLE[run["model_key"]])
    if spec["kind"] == "qlora" and (why := qlora_blocker()):
        return dict(rec, status="skipped", reason=why)
    if spec["kind"] in ("lora", "qlora") and not spec.get("hp"):
        return dict(rec, status="skipped", reason="no finished stage-1 search trial to take LoRA hyperparameters from")
    out, oom = None, False
    try:
        out = train_one(run["model_key"], spec, run["seed"], BATCH_SIZE, 1)
    except RuntimeError as e:                              # includes torch.cuda.OutOfMemoryError
        if "out of memory" not in str(e).lower():
            return dict(rec, status="failed", reason=f"{type(e).__name__}: {str(e)[:300]}", trace=traceback.format_exc()[-1200:])
        oom = True
    except Exception as e:
        return dict(rec, status="failed", reason=f"{type(e).__name__}: {str(e)[:300]}", trace=traceback.format_exc()[-1200:])
    if oom:
        free_memory()
        print("    out of memory -> retrying with half the batch size and 2x gradient accumulation")
        try:
            out = train_one(run["model_key"], spec, run["seed"], max(1, BATCH_SIZE // 2), 2)
        except Exception as e:
            free_memory()
            return dict(rec, status="failed", reason=f"OOM even after halving batch: {str(e)[:200]}")
    result, preds = out
    preds.to_csv(OUT_DIR / "predictions" / f"{run['run_name']}__{run['config_hash']}.csv", index=False)
    return dict(rec, status="ok", **result, env=ENV_INFO, finished_at=dt.datetime.now().isoformat(timespec="seconds"))


def run_list(runs, label, t0):
    todo = [r for r in runs if not load_ok(r)]
    print(f"{label}: {len(runs)} runs planned, {len(runs) - len(todo)} already finished, {len(todo)} to do.")
    for i, run in enumerate(todo, 1):
        print(f"[{label} {i}/{len(todo)}] {run['run_name']}  (elapsed {(time.time() - t0) / 60:.1f} min)")
        rec = execute(run)
        free_memory()                                         # release GPU memory before the next run
        write_json(result_path(run["run_name"], run["config_hash"]), rec)
        log_mlflow(rec)
        if rec["status"] == "ok":
            m = rec["metrics"]
            print(f"    -> val F1 {m['val']['macro_f1']:.3f} (against {m['val']['per_class']['against']['f1']:.3f}) | "
                  f"trainable {rec['trainable_params']:,} ({rec['trainable_pct']:.2f}%) | {rec['train_time_sec']:.0f}s train")
        else:
            print(f"    -> {rec['status'].upper()}: {rec['reason']}")


def run_all():
    global PLANNED, CHOSEN_HP, SEARCH_TABLE, ACTIVE_HASHES
    t0 = time.time()
    if not result_path("baseline__majority", BASE_HASH).exists():
        print("Fitting baselines ...")
        for rec in run_baselines(BASE_HASH):
            write_json(result_path(rec["run_name"], BASE_HASH), rec)
            log_mlflow(rec)

    print(f"Stage 1: LoRA search, {len(lora_grid())} combinations x seeds {SEARCH_SEEDS} for "
          f"{[m for m, _, _ in searched_methods()]} on each model")
    run_list(SEARCH_PLAN, "search", t0)
    CHOSEN_HP, SEARCH_TABLE = choose_hp(SEARCH_PLAN)
    for (mk, mid), hp in sorted(CHOSEN_HP.items()):
        print(f"    picked for {mk} / {mid}: {hp_tag(hp)}")

    PLANNED = plan_runs(CHOSEN_HP)
    ACTIVE_HASHES = {r["config_hash"] for r in SEARCH_PLAN + PLANNED} | {BASE_HASH}
    print("Stage 2: method comparison")
    run_list(PLANNED, "compare", t0)
    print(f"Done in {(time.time() - t0) / 60:.1f} min.")


run_all()


MLflow store: /content/drive/MyDrive/EcoConsensus/experiment_outputs/mlflow.db  (artifacts: /content/drive/MyDrive/EcoConsensus/experiment_outputs/mlartifacts)
Fitting baselines ...
Stage 1: LoRA search, 36 combinations x seeds [1] for ['lora'] on each model
search: 72 runs planned, 0 already finished, 72 to do.
[search 1/72] muril__search__lora__r8_a8_qv_lr0.0003__seed1  (elapsed 0.3 min)


W1109 18:54:58.078000 12147 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1109 18:54:58.254000 12147 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google/muril-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params w

    epoch 1: loss 1.097 | val F1 0.241 | val acc 0.567 | 13s
    epoch 2: loss 1.065 | val F1 0.551 | val acc 0.690 | 4s
    epoch 3: loss 0.887 | val F1 0.594 | val acc 0.733 | 4s
    -> val F1 0.594 (against 0.242) | trainable 887,811 (0.37%) | 20s train
[search 2/72] muril__search__lora__r8_a8_qv_lr0.001__seed1  (elapsed 0.9 min)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

... (268 outputs left out)

    epoch 1: loss 0.958 | val F1 0.616 | val acc 0.762 | 4s
    epoch 2: loss 0.246 | val F1 0.662 | val acc 0.819 | 4s
    epoch 3: loss 0.042 | val F1 0.635 | val acc 0.793 | 4s
    -> val F1 0.662 (against 0.311) | trainable 1,182,723 (0.42%) | 12s train
[compare 20/20] indicbert__qlora_4bit__seed43  (elapsed 27.9 min)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: ai4bharat/IndicBERTv2-MLM-only
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those p

    epoch 1: loss 0.938 | val F1 0.589 | val acc 0.695 | 7s
    epoch 2: loss 0.287 | val F1 0.639 | val acc 0.805 | 7s
    epoch 3: loss 0.051 | val F1 0.624 | val acc 0.786 | 6s
    -> val F1 0.639 (against 0.246) | trainable 1,182,723 (0.42%) | 20s train
Done in 28.4 min.


## 9. Report
A new file `reports/experiment_report_<timestamp>.md` (plus `_runs.csv`) is written to the outputs folder on Drive every time this cell runs; older reports are never overwritten. The report is also attached to MLflow as an artifact.

In [9]:
# ============================ 9. Report: plain Python, no AI. Every run creates a NEW timestamped file. ============================
def load_results():
    recs = []
    for p in sorted(RESULTS_DIR.glob("*.json")):
        try:
            r = json.loads(p.read_text(encoding="utf-8"))
        except Exception:
            continue
        if r.get("config_hash") in ACTIVE_HASHES:      # ignore results made with other settings
            recs.append(r)
    return recs


def _vals(recs, getter):
    out = []
    for r in recs:
        try:
            v = getter(r)
        except (KeyError, TypeError):
            v = None
        if v is not None:
            out.append(v)
    return out


def mean_std(vals):
    if not vals:
        return None, None
    return float(np.mean(vals)), (float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0)


def fmt_ms(vals, nd=3):
    m, s = mean_std(vals)
    if m is None:
        return "-"
    return f"{m:.{nd}f}" if len(vals) < 2 else f"{m:.{nd}f} ± {s:.{nd}f}"


def md_table(headers, rows):
    lines = ["| " + " | ".join(headers) + " |", "|" + "|".join("---" for _ in headers) + "|"]
    lines += ["| " + " | ".join(str(c) for c in r) + " |" for r in rows]
    return "\n".join(lines)


REFERENCES = """\
- **MuRIL**: Khanuja et al., 2021, *MuRIL: Multilingual Representations for Indian Languages*, Google Research. arXiv:2103.10730. Model: `google/muril-base-cased` (Google).
- **IndicBERT v2**: Doddapaneni et al., 2023, *Towards Leaving No Indic Language Behind: Building Monolingual Corpora, Benchmark and Models for Indic Languages*, ACL 2023 (AI4Bharat). Model: `ai4bharat/IndicBERTv2-MLM-only`.
- **Transformers**: Wolf et al., 2020, *Transformers: State-of-the-Art Natural Language Processing*, EMNLP 2020 (Hugging Face).
- **PEFT** (LoRA/QLoRA adapters): Hugging Face PEFT library (Mangrulkar et al., 2022).
- **LoRA**: Hu et al., 2021, *LoRA: Low-Rank Adaptation of Large Language Models*, Microsoft. arXiv:2106.09685.
- **QLoRA**: Dettmers et al., 2023, *QLoRA: Efficient Finetuning of Quantized LLMs*, NeurIPS 2023; **bitsandbytes** by Tim Dettmers et al.
- **AdamW**: Loshchilov and Hutter, 2019, *Decoupled Weight Decay Regularization*, ICLR 2019.
- **PyTorch**: Paszke et al., 2019, *PyTorch: An Imperative Style, High-Performance Deep Learning Library*, NeurIPS 2019.
- **scikit-learn**: Pedregosa et al., 2011, JMLR 12:2825-2830.
- **MLflow**: Zaharia et al., 2018, *Accelerating the Machine Learning Lifecycle with MLflow*, IEEE Data Engineering Bulletin 41(4) (Databricks).
- **SemEval-2016 Task 6** (experiment data): Mohammad, Kiritchenko, Sobhani, Zhu and Cherry, 2016, *SemEval-2016 Task 6: Detecting Stance in Tweets*, SemEval-2016.
- Hindi and Bengali text is machine-translated from the English tweets with an LLM (see the project README for the model used).
"""


def generate_report():
    recs = load_results()
    ok = [r for r in recs if r.get("kind") == "transformer" and r["status"] == "ok"]
    bad = [r for r in recs if r.get("kind") in ("transformer", "search") and r["status"] != "ok"]
    base = {r["method_id"]: r for r in recs if r.get("kind") == "baseline"}
    order = {(mk, m): i for i, (mk, m) in enumerate((mk, m) for mk in MODELS for m in method_ids())}
    groups = {}
    for r in ok:
        groups.setdefault((r["model_key"], r["method_id"]), []).append(r)
    keys = sorted(groups, key=lambda k: order.get(k, 999))

    def stat(k, split, metric):
        return _vals(groups[k], lambda r: r["metrics"][split][metric])

    def params_of(k):
        return float(np.mean([r["trainable_params"] for r in groups[k]]))

    def label(k):                                   # method id plus its LoRA hyperparameters, if any
        hp = groups[k][0].get("hp")
        return f"{k[1]} ({hp_tag(hp)})" if hp else k[1]

    # ---- choose per model ----
    picks = {}
    for mk in MODELS:
        ks = [k for k in keys if k[0] == mk]
        if not ks:
            continue
        means = {k: mean_std(stat(k, RANK_SET, "macro_f1")) for k in ks}
        best = max(ks, key=lambda k: means[k][0])
        tol = max(means[best][1] or 0.0, 0.01)
        close = [k for k in ks if means[k][0] >= means[best][0] - tol]
        rec = min(close, key=params_of)
        picks[mk] = dict(best=best, rec=rec, tol=tol, close=close, means=means)

    now = dt.datetime.now()
    L = []
    L.append(f"# Fine-tuning method comparison report\n\nGenerated {now:%Y-%m-%d %H:%M:%S} by the experiment notebook "
             f"(deterministic script, no AI). Experiment: `{EXPERIMENT_NAME}`. Settings hash set: {len(ACTIVE_HASHES)} runs/baselines.\n")

    # ---- summary ----
    L.append("## 1. Summary\n")
    L.append(f"Methods are **ranked on `{RANK_SET}`** (validation split of the SemEval subset, Macro-F1, mean ± sample std over seeds). The best epoch inside each run "
             f"is chosen on `{EARLY_STOP_MONITOR}`. The held-out `test` split is shown only for the recommended methods (section 5) and never used to rank.\n")
    if not picks:
        L.append("**No transformer run finished successfully yet.** See section 8.\n")
    for mk, p in picks.items():
        b, r = p["best"], p["rec"]
        L.append(f"- **{mk}**: highest `{RANK_SET}` Macro-F1 = **{label(b)}** ({fmt_ms(stat(b, RANK_SET, 'macro_f1'))}). "
                 f"Recommended by the parsimony rule = **{label(r)}** ({fmt_ms(stat(r, RANK_SET, 'macro_f1'))}, "
                 f"{params_of(r):,.0f} trainable parameters). Methods within {p['tol']:.3f} of the best: {', '.join(k[1] for k in p['close'])}.")
    if picks and "tfidf_lr" in base:
        tf = base["tfidf_lr"]["metrics"][RANK_SET]["macro_f1"]
        best_tr = max(p["means"][p["best"]][0] for p in picks.values())
        verdict = "beats" if best_tr > tf else "does NOT beat"
        L.append(f"- Reference: TF-IDF + LR scores {tf:.3f} on `{RANK_SET}`; the best transformer method ({best_tr:.3f}) {verdict} it on this set.")
    L.append("\n**Parsimony rule:** take the method with the best mean score, then, among all methods of the same model whose mean is within "
             "max(1 std of the best, 0.010) of it, recommend the one with the fewest trainable parameters. Differences inside that band are noise on this dataset size.\n")

    # ---- data ----
    L.append("## 2. Data\n")
    L.append(f"SemEval-2016 Task 6 subset: {MANIFEST['datasets']['semeval']['source_rows']} of {MANIFEST['datasets']['semeval']['source_rows_available']} tweets "
             f"(all tweets of {MANIFEST.get('keep_all_classes') or 'no class'} kept, the rest stratified), each in English + 3 Hindi + 3 Bengali variants. "
             f"Frozen split `{SPLIT_KEY}` (seed {SPLIT_SEED}), split by tweet: train {TRAIN_FRACTION} / val {VAL_FRACTION} / test {1 - TRAIN_FRACTION - VAL_FRACTION:.2f}; "
             f"training cap {TRAIN_SOURCE_ROWS or 'none'}, variant policy `{TRAIN_VARIANT_POLICY}`, claim mode `{CLAIM_MODE}`. "
             f"Labels: {', '.join(f'{i} = {l}' for i, l in enumerate(LABELS))}. Rows dropped for empty comments: {DROPPED}.\n")
    L.append(_report_table(SPLITS) + "\n")
    for note in MANIFEST.get("notes", []):
        L.append(f"> Data note: {note}\n")
    L.append("Token lengths (sentence pair) and share of `[UNK]` tokens per language group:\n")
    rows = [[k, s["model"], s["p50"], s["p95"], s["p99"], s["max"], f"{s['truncated_pct_at_max_len']}%",
             ", ".join(f"{lg}: {v:.2%}" for lg, v in s["unk_rate_by_language"].items()) or "-"]
            for k, s in TOKEN_STATS.items()]
    L.append(md_table(["model", "checkpoint", "p50", "p95", "p99", "max", f"truncated at {MAX_LEN}", "[UNK] rate"], rows) + "\n")

    # ---- main comparison ----
    L.append("## 3. Method comparison\n")
    L.append(f"All numbers: mean ± sample std over seeds {SEEDS} (n = seeds that finished). Rank set = `{RANK_SET}`.\n")
    rows = []
    for mk in MODELS:
        ks = sorted([k for k in keys if k[0] == mk], key=lambda k: -(mean_std(stat(k, RANK_SET, 'macro_f1'))[0] or 0))
        for rank, k in enumerate(ks, 1):
            g = groups[k]
            mark = " (best)" if picks[mk]["best"] == k else ""
            mark += " **(recommended)**" if picks[mk]["rec"] == k else ""
            rows.append([mk, f"{label(k)}{mark}", rank, len(g), fmt_ms(stat(k, RANK_SET, "macro_f1")),
                         fmt_ms(stat(k, RANK_SET, "accuracy")),
                         fmt_ms(_vals(g, lambda r: r["metrics"][RANK_SET]["per_class"]["against"]["f1"])),
                         f"{params_of(k):,.0f}", f"{np.mean([r['trainable_pct'] for r in g]):.2f}%",
                         f"{np.mean([r['train_time_sec'] for r in g]):.0f}",
                         "-" if not _vals(g, lambda r: r['peak_mem_mb']) else f"{np.mean(_vals(g, lambda r: r['peak_mem_mb'])):.0f}",
                         f"{np.mean([r['best_epoch'] for r in g]):.1f}",
                         " / ".join(f"{v:.3f}" for v in stat(k, RANK_SET, "macro_f1"))])
    L.append(md_table(["model", "method", "rank", "n", f"{RANK_SET} Macro-F1", f"{RANK_SET} accuracy", f"{RANK_SET} F1 against",
                       "trainable params", "% trainable", "train sec", "peak mem MB", "best epoch", "per-seed F1"], rows) + "\n")

    # ---- stage-1 LoRA search ----
    L.append("### LoRA / QLoRA hyperparameter search (stage 1)\n")
    L.append(f"Grid `{LORA_SEARCH}` ({len(lora_grid())} combinations{', random sample' if SEARCH_MAX_TRIALS else ''}), seeds {SEARCH_SEEDS}, "
             f"scored on `val`. alpha = r x alpha_mult. Target sets: `{LORA_TARGET_SETS}`. Pick rule: best mean val Macro-F1, then the fewest "
             "trainable parameters within max(1 std, 0.010) of it. Stage 2 (table above) re-runs the pick with seeds "
             f"{SEEDS}. QLoRA hyperparameters: `{QLORA_HPARAMS}`.\n")
    if not SEARCH_TABLE:
        L.append("No finished search trials.\n")
    for (mk, mid), t in sorted(SEARCH_TABLE.items(), key=lambda kv: order.get(kv[0], 999)):
        L.append(f"**{mk} / {mid}**: best `{t['best']}`, picked **`{t['pick']}`** (band {t['tol']:.3f}).\n")
        rows = [[x["tag"] + (" **(picked)**" if x["tag"] == t["pick"] else ""), x["hp"]["r"], x["hp"]["r"] * x["hp"]["alpha_mult"],
                 x["hp"]["targets"], f"{x['hp']['lr']:g}", f"{x['params']:,}", x["n"],
                 f"{x['mean']:.3f}" + (f" ± {x['std']:.3f}" if x["n"] > 1 else ""), f"{x['against']:.3f}"] for x in t["rows"]]
        L.append(md_table(["combination", "r", "alpha", "targets", "lr", "trainable params", "n", "val Macro-F1", "val F1 against"], rows) + "\n")

    # ---- baselines ----
    L.append("## 4. Reference baselines (same training split, same sets)\n")
    rows = [[n, b["params"], f"{b['metrics']['val']['macro_f1']:.3f}", f"{b['metrics']['val']['accuracy']:.3f}",
             f"{b['metrics']['val']['per_class']['against']['f1']:.3f}"] for n, b in base.items()]
    L.append(md_table(["baseline", "settings", "val Macro-F1", "val accuracy", "val F1 against"], rows) + "\n")

    # ---- held-out check ----
    L.append("## 5. Held-out check on `test` (recommended method per model)\n")
    L.append("Computed after the choice was made by the rules above. Do not go back and change the choice because of these numbers.\n")
    rows, conf_blocks = [], []
    for mk, p in picks.items():
        k = p["rec"]
        rows.append([f"{mk} / {k[1]}"] + [fmt_ms(stat(k, "test", m)) for m in ("macro_f1", "accuracy", "weighted_f1", "mcc", "kappa")]
                    + [fmt_ms(_vals(groups[k], lambda r, c=c: r["metrics"]["test"]["per_class"][c]["f1"])) for c in LABELS])
        cm = np.sum([np.array(r["metrics"]["test"]["confusion"]) for r in groups[k]], axis=0)
        conf_blocks.append(f"Confusion matrix, {mk} / {k[1]} (rows = true, columns = predicted, summed over {len(groups[k])} seed(s)):\n\n"
                           + md_table(["true \\ pred"] + LABELS, [[LABELS[i]] + [int(x) for x in cm[i]] for i in range(3)]) + "\n")
    for n, b in base.items():
        m = b["metrics"]["test"]
        rows.append([f"baseline / {n}", f"{m['macro_f1']:.3f}", f"{m['accuracy']:.3f}", f"{m['weighted_f1']:.3f}",
                     "-" if m['mcc'] is None else f"{m['mcc']:.3f}", "-" if m['kappa'] is None else f"{m['kappa']:.3f}"]
                    + [f"{m['per_class'][c]['f1']:.3f}" for c in LABELS])
    L.append(md_table(["system", "Macro-F1", "accuracy", "weighted-F1", "MCC", "kappa"] + [f"F1 {c}" for c in LABELS], rows) + "\n")
    L += conf_blocks

    # ---- per language ----
    L.append(f"## 6. Macro-F1 by language and by script variant (`{RANK_SET}`)\n")
    langs = sorted(SPLITS[RANK_SET]["language"].unique())
    rows = []
    for k in keys:
        rows.append([k[0], k[1]] + [fmt_ms(_vals(groups[k], lambda r, lg=lg: r["metrics"][RANK_SET]["by_language"][lg])) for lg in langs]
                    + [fmt_ms(_vals(groups[k], lambda r, lg=lg: r["metrics"]["test"]["by_language"][lg])) if picks.get(k[0], {}).get("rec") == k else "-" for lg in langs])
    for n, b in base.items():
        rows.append(["baseline", n] + [f"{b['metrics'][RANK_SET]['by_language'][lg]:.3f}" for lg in langs] + ["-" for _ in langs])
    L.append(md_table(["model", "method"] + [f"{RANK_SET} {lg}" for lg in langs] + [f"test {lg} (recommended only)" for lg in langs], rows) + "\n")
    variants = sorted((SPLITS[RANK_SET]["language"] + "/" + SPLITS[RANK_SET]["variant"]).unique())
    rows = [[k[0], k[1]] + [fmt_ms(_vals(groups[k], lambda r, v=v: r["metrics"][RANK_SET]["by_variant"][v])) for v in variants] for k in keys]
    rows += [["baseline", n] + [f"{b['metrics'][RANK_SET]['by_variant'][v]:.3f}" for v in variants] for n, b in base.items()]
    L.append(f"By script variant (`{RANK_SET}`; each column has the same tweets, so differences come from the text form):\n")
    L.append(md_table(["model", "method"] + variants, rows) + "\n")

    # ---- learning curves ----
    L.append("## 7. Learning curves (first seed of each method)\n")
    rows = []
    for k in keys:
        r = sorted(groups[k], key=lambda r: r["seed"])[0]
        h = r["history"]
        rows.append([k[0], label(k), r["seed"], " / ".join(f"{x['train_loss']:.3f}" for x in h),
                     " / ".join(f"{x['val_macro_f1']:.3f}" for x in h), " / ".join(f"{x['val_accuracy']:.3f}" for x in h)])
    L.append(md_table(["model", "method", "seed", "train loss per epoch", "val Macro-F1 per epoch", "val accuracy per epoch"], rows) + "\n")

    # ---- failures ----
    L.append("## 8. Skipped / failed runs\n")
    if bad:
        L.append(md_table(["run", "status", "reason"], [[r["run_name"], r["status"], str(r.get("reason", "")).replace("|", "/")] for r in bad]) + "\n")
        L.append("Skipped or failed methods are reported as **not run**; they are not scored as zero.\n")
    else:
        L.append("None.\n")

    # ---- caveats ----
    L.append("## 9. How to read these results\n")
    n_val = SPLITS[RANK_SET]["source_id"].nunique()
    n_against = SPLITS[RANK_SET].drop_duplicates("source_id")["label"].eq("against").sum()
    L.append("- Train, val and test all come from the same SemEval subset (tweets), so these are in-domain scores. They are for **choosing a method**, "
             "not final results: the final models are trained as described in docs/TRD.md and evaluated separately.\n"
             "- Hindi and Bengali rows are machine translations of the English tweets; each language has 3 script variants (pure, romanised, code-mixed). "
             "Per-language and per-variant scores partly reflect translation quality and tokenizer coverage (see the `[UNK]` rates in section 2).\n"
             f"- `{RANK_SET}` has {n_val} tweets ({n_against} labelled against), each in 7 versions, and the 7 versions are not independent. "
             "Differences of a few points are within noise; prefer the parsimony pick when scores are close.\n"
             "- `against` is rare, so its F1, and therefore Macro-F1, moves a lot with a handful of tweets. Check the per-class columns and the confusion matrix.\n"
             f"- The best epoch and the method ranking both use `{RANK_SET}`, so its scores are slightly optimistic; `test` (section 5) is the unbiased check.\n"
             f"- LoRA / QLoRA hyperparameters (r, alpha, target modules, learning rate) were picked by the stage-1 search on `{RANK_SET}`; "
             "head-only and last-N use fixed learning rates. Stage 2 re-runs the pick with new seeds, but since the combination itself was "
             f"chosen on `{RANK_SET}`, LoRA-type scores there are a little optimistic compared with the others. `test` (section 5) is the fair check.\n")

    # ---- environment / config ----
    L.append("## 10. Environment and configuration\n")
    envs = {json.dumps(r.get("env", {}), sort_keys=True) for r in ok} or {json.dumps(ENV_INFO, sort_keys=True)}
    for e in envs:
        L.append("```json\n" + json.dumps(json.loads(e), indent=2) + "\n```\n")
    cfg_names = ["MODELS", "RUN_METHODS", "LAST_N_LIST", "LR", "LORA_SEARCH", "LORA_TARGET_SETS", "LORA_DROPOUT", "SEARCH_SEEDS",
                 "SEARCH_MAX_TRIALS", "QLORA_BITS", "QLORA_HPARAMS", "CLAIM_MODE", "FIXED_CLAIM",
                 "TRAIN_FRACTION", "VAL_FRACTION", "TRAIN_SOURCE_ROWS", "TRAIN_VARIANT_POLICY", "SPLIT_SEED", "SEEDS", "EPOCHS", "BATCH_SIZE", "MAX_LEN", "WEIGHT_DECAY",
                 "WARMUP_FRACTION", "GRAD_CLIP", "HEAD_DROPOUT", "PATIENCE", "EARLY_STOP_MONITOR", "RANK_SET"]
    L.append("```json\n" + json.dumps({n: globals()[n] for n in cfg_names}, indent=2, default=str) + "\n```\n")
    L.append(f"Total training+evaluation time of the runs in this report: {sum(r['total_time_sec'] for r in ok) / 60:.1f} min. "
             f"MLflow store: `{MLFLOW_DB_OUT}` (copy it to the Mac and open with `mlflow ui --backend-store-uri sqlite:///mlflow.db`).\n")
    L.append("## 11. References\n")
    L.append(REFERENCES)

    # ---- write: never overwrite ----
    stamp = now.strftime("%Y%m%d_%H%M%S")
    n = 0
    while True:
        stem = f"experiment_report_{stamp}" + (f"_{n}" if n else "")
        path = OUT_DIR / "reports" / f"{stem}.md"
        try:
            with open(path, "x", encoding="utf-8") as f:
                f.write("\n".join(L))
            break
        except FileExistsError:
            n += 1
    flat = [dict(run_name=r["run_name"], model=r["model_key"], method=r["method_id"], hp=hp_tag(r.get("hp")), seed=r["seed"],
                 val_macro_f1=r["metrics"]["val"]["macro_f1"], val_accuracy=r["metrics"]["val"]["accuracy"],
                 val_f1_against=r["metrics"]["val"]["per_class"]["against"]["f1"], trainable_params=r["trainable_params"],
                 trainable_pct=r["trainable_pct"], train_time_sec=r["train_time_sec"], peak_mem_mb=r["peak_mem_mb"],
                 best_epoch=r["best_epoch"]) for r in ok]
    pd.DataFrame(flat).to_csv(OUT_DIR / "reports" / f"{stem}_runs.csv", index=False)
    return path


REPORT_PATH = generate_report()
print("Report written to:", REPORT_PATH)
if MLFLOW_OK:                                   # attach the report to MLflow too (one small run per report)
    try:
        with mlflow.start_run(run_name=f"report__{REPORT_PATH.stem}"):
            mlflow.set_tag("kind", "report")
            mlflow.log_artifact(str(REPORT_PATH))
            mlflow.log_artifact(str(REPORT_PATH.with_name(REPORT_PATH.stem + "_runs.csv")))
    except Exception as e:
        print(f"[mlflow report log failed] {type(e).__name__}: {str(e)[:150]}")
    sync_mlflow()
print("\n".join(REPORT_PATH.read_text(encoding="utf-8").splitlines()[:40]))


Report written to: /content/drive/MyDrive/EcoConsensus/experiment_outputs/reports/experiment_report_20261109_192255.md
# Fine-tuning method comparison report

Generated 2026-11-09 19:22:55 by the experiment notebook (deterministic script, no AI). Experiment: `finetune_method_comparison`. Settings hash set: 93 runs/baselines.

## 1. Summary

Methods are **ranked on `val`** (validation split of the SemEval subset, Macro-F1, mean ± sample std over seeds). The best epoch inside each run is chosen on `val`. The held-out `test` split is shown only for the recommended methods (section 5) and never used to rank.

- **muril**: highest `val` Macro-F1 = **lora (r16_a32_qkv_lr0.001)** (0.632 ± 0.011). Recommended by the parsimony rule = **lora (r16_a32_qkv_lr0.001)** (0.632 ± 0.011, 1,477,635 trainable parameters). Methods within 0.011 of the best: lora.
- **indicbert**: highest `val` Macro-F1 = **qlora_4bit (r16_a32_qv_lr0.001)** (0.654 ± 0.021). Recommended by the parsimony rule = **lora (r16_a3